# Attention from scratch
**LLMs, Animated · M1.1 hands-on** &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhishekaryan23/ai-dev-tasks/blob/main/course/notebooks/m1-1-attention.ipynb)

In the lesson you watched one token, *bank*, attend over a sentence. Here you'll build the same machinery in about 20 lines of NumPy:

1. a numerically stable **softmax**
2. **scaled dot-product attention**, $\mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V$
3. the **causal mask** $M$
4. a check against PyTorch's built-in version
5. an attention **heatmap**
6. an experiment showing **why we divide by $\sqrt{d_k}$**

**How to use it:** run cells top to bottom (Shift+Enter). Each `# TODO` cell is followed by a ✅ check cell. Stuck for more than five minutes? The **Solutions** are at the bottom. Try first: the struggle is what makes it stick.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## 1. A numerically stable softmax
$\mathrm{softmax}(x)_i = \dfrac{e^{x_i}}{\sum_j e^{x_j}}$. Computed naively, $e^{1000}$ overflows to `inf`.
The trick: subtracting the same constant from every entry doesn't change the result, so subtract the **max** first. The largest exponent becomes $e^0 = 1$.

In [ ]:
def softmax(x, axis=-1):
    """Softmax along `axis`. Must not overflow for large inputs."""
    # TODO: subtract the max along `axis` (use keepdims=True), exponentiate,
    #       then divide by the sum along the same axis.
    raise NotImplementedError("Fill in softmax")

In [ ]:
p = softmax(np.array([1.0, 2.0, 3.0]))
assert np.allclose(p, [0.0900, 0.2447, 0.6652], atol=1e-4), p
assert np.allclose(softmax(np.array([1000.0, 1001.0])), [0.2689, 0.7311], atol=1e-4), "Overflow? Subtract the max first."
M = rng.normal(size=(4, 5))
assert np.allclose(softmax(M, axis=-1).sum(axis=-1), 1), "Each row should sum to 1."
print("✅ softmax works:", p)

## 2. Scaled dot-product attention
Shapes for $n$ queries attending over $m$ keys:

| | shape | meaning |
|---|---|---|
| `Q` | $(n, d_k)$ | what each token is looking for |
| `K` | $(m, d_k)$ | what each token offers |
| `V` | $(m, d_v)$ | what each token passes on |
| `Q @ K.T` | $(n, m)$ | one score per (query, key) pair |

Fill in `attention`. It returns both the output and the weights, so we can look at them.

In [ ]:
def attention(Q, K, V, mask=None):
    """Q: (n, d_k), K: (m, d_k), V: (m, d_v). mask: additive (n, m) array of 0 / -inf, or None.
    Returns (output (n, d_v), weights (n, m))."""
    # TODO 1: scores = Q @ K.T divided by sqrt(d_k)   (d_k = Q.shape[-1])
    # TODO 2: if a mask is given, add it to the scores
    # TODO 3: weights = softmax of the scores over the last axis
    # TODO 4: return weights @ V, weights
    raise NotImplementedError("Fill in attention")

In [ ]:
# The worked example from the lesson: q = [1, 2], keys = values = [1,0], [0,1], [1,1]
K = np.array([[1., 0.], [0., 1.], [1., 1.]]); V = K.copy()
out, w = attention(np.array([[1., 2.]]), K, V)
assert np.allclose(w, [[0.140, 0.284, 0.576]], atol=1e-3), w
assert np.allclose(out, [[0.716, 0.860]], atol=1e-3), out

# The "your turn" example: q = [2, 0]. Compare with your answers in the lesson!
out2, w2 = attention(np.array([[2., 0.]]), K, V)
print("your-turn weights:", w2.round(3), " output:", out2.round(3))

Q = rng.normal(size=(6, 8)); Kr = rng.normal(size=(6, 8)); Vr = rng.normal(size=(6, 4))
o, w = attention(Q, Kr, Vr)
assert o.shape == (6, 4) and w.shape == (6, 6), (o.shape, w.shape)
assert np.allclose(w.sum(-1), 1)
print("✅ attention works")

## 3. The causal mask
A decoder LLM predicts the *next* token, so token $i$ may only look at tokens $j \le i$.
We add $0$ to the allowed scores and $-\infty$ to the future ones; since $e^{-\infty} = 0$, the future gets exactly zero weight.

In [ ]:
def causal_mask(n):
    """Additive mask of shape (n, n): 0 where key j <= query i, -inf where j > i."""
    # TODO: np.triu(np.ones((n, n), dtype=bool), k=1) is True exactly at the future positions.
    #       Use np.where to put -np.inf there and 0.0 everywhere else.
    raise NotImplementedError("Fill in causal_mask")

In [ ]:
m = causal_mask(4)
print(m)
assert m.shape == (4, 4)
assert np.all(m[np.tril_indices(4)] == 0), "On and below the diagonal should be 0."
assert np.all(np.isneginf(m[np.triu_indices(4, k=1)])), "Above the diagonal should be -inf."

o, w = attention(Q, Kr, Vr, mask=causal_mask(6))
assert np.allclose(np.triu(w, k=1), 0), "Future tokens must get zero weight."
assert np.allclose(w.sum(-1), 1) and np.allclose(w[0], [1, 0, 0, 0, 0, 0]), "The first token can only see itself."
print("✅ causal mask works")

## 4. Check against PyTorch (optional)
PyTorch ships an optimized version, `torch.nn.functional.scaled_dot_product_attention`. On a GPU it can dispatch to FlashAttention, which you'll meet in M5.4. Your NumPy version should match it.

In [ ]:
try:
    import torch
    import torch.nn.functional as F
    Qt, Kt, Vt = (torch.tensor(a, dtype=torch.float64)[None] for a in (Q, Kr, Vr))  # add a batch dim
    ref = F.scaled_dot_product_attention(Qt, Kt, Vt, is_causal=True)[0].numpy()
    mine, _ = attention(Q, Kr, Vr, mask=causal_mask(6))
    print("max |difference| vs PyTorch:", float(np.abs(ref - mine).max()))
    assert np.allclose(ref, mine, atol=1e-6)
    print("✅ matches torch.nn.functional.scaled_dot_product_attention")
except ImportError:
    print("PyTorch isn't installed here, so skipping this check. (It comes preinstalled on Colab.)")

## 5. A tiny self-attention layer, as a heatmap
Real models get $X$ from an embedding table and *learn* $W_Q, W_K, W_V$. Ours are random (untrained), so the pattern below is arbitrary.
Still, check the structure: the upper triangle is empty (masked) and every row sums to 1.

In [ ]:
tokens = ["I", "sat", "by", "the", "river", "bank"]
d_model, d_k = 16, 8
X = rng.normal(size=(len(tokens), d_model))                       # stand-in embeddings
W_Q, W_K, W_V = (rng.normal(size=(d_model, d_k)) / np.sqrt(d_model) for _ in range(3))

out, weights = attention(X @ W_Q, X @ W_K, X @ W_V, mask=causal_mask(len(tokens)))
print("output shape:", out.shape, "| row sums:", weights.sum(-1).round(3))

fig, ax = plt.subplots(figsize=(5.2, 4.6))
im = ax.imshow(weights, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(tokens)), tokens); ax.set_yticks(range(len(tokens)), tokens)
ax.set_xlabel("key (looked at)"); ax.set_ylabel("query (looking)")
for i in range(len(tokens)):
    for j in range(i + 1):
        ax.text(j, i, f"{weights[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if weights[i, j] > 0.5 else "black")
fig.colorbar(im, ax=ax, fraction=0.046)
ax.set_title("Causal self-attention weights (untrained)")
plt.tight_layout(); plt.show()

## 6. Experiment: why divide by $\sqrt{d_k}$?
Draw a random query and 16 random keys with unit-variance entries, and measure how much weight the **top** key gets, averaged over many trials.
Run it and compare the two columns.

In [ ]:
def avg_top_weight(d_k, scale, n_keys=16, trials=300):
    tops = []
    for _ in range(trials):
        q = rng.normal(size=(1, d_k)); K = rng.normal(size=(n_keys, d_k))
        s = q @ K.T
        if scale:
            s = s / np.sqrt(d_k)
        tops.append(softmax(s).max())
    return float(np.mean(tops))

dks = [4, 16, 64, 256, 1024]
unscaled = [avg_top_weight(d, False) for d in dks]
scaled = [avg_top_weight(d, True) for d in dks]
for d, u, s in zip(dks, unscaled, scaled):
    print(f"d_k={d:5d}   top weight without scaling={u:.2f}   with scaling={s:.2f}")

plt.figure(figsize=(5.4, 3.4))
plt.semilogx(dks, unscaled, "o-", label="q·k (no scaling)")
plt.semilogx(dks, scaled, "o-", label="q·k / √d_k")
plt.axhline(1 / 16, ls=":", c="gray", label="uniform (1/16)")
plt.xlabel("d_k"); plt.ylabel("average top attention weight"); plt.ylim(0, 1.05)
plt.legend(); plt.tight_layout(); plt.show()

assert unscaled[-1] > 0.9, "Without scaling, big d_k should make softmax nearly one-hot."
assert max(scaled) - min(scaled) < 0.15, "With scaling, the top weight should barely change with d_k."
print("✅ Without scaling the softmax saturates as d_k grows; with scaling it stays put.")

**Think about it** (answers are in the Solutions section):
1. When softmax is almost one-hot, what happens to the gradient that flows to the *other* keys?
2. Why is the variance of $q \cdot k$ about $d_k$ when the entries have variance 1?
3. Your `attention` builds an $n \times n$ matrix. How much memory does that take for $n = 100{,}000$ in float32, per head?

---
## Solutions
Only run these if you're stuck. They redefine the three functions; re-run the check cells afterwards.

In [ ]:
def softmax(x, axis=-1):
    """Softmax along `axis`. Must not overflow for large inputs."""
    z = x - np.max(x, axis=axis, keepdims=True)   # shift so the largest entry is 0
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

In [ ]:
def attention(Q, K, V, mask=None):
    """Q: (n, d_k), K: (m, d_k), V: (m, d_v). mask: additive (n, m) array of 0 / -inf, or None.
    Returns (output (n, d_v), weights (n, m))."""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = scores + mask
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

In [ ]:
def causal_mask(n):
    """Additive mask of shape (n, n): 0 where key j <= query i, -inf where j > i."""
    future = np.triu(np.ones((n, n), dtype=bool), k=1)
    return np.where(future, -np.inf, 0.0)

**Think-about-it answers**
1. The softmax Jacobian is $\partial p_i / \partial s_j = p_i(\delta_{ij} - p_j)$. If one $p \approx 1$ and the rest are $\approx 0$, every term is close to 0, so almost no gradient reaches the scores, and $W_Q, W_K$ barely learn.
2. $q\cdot k = \sum_{i=1}^{d_k} q_i k_i$. Each product of independent zero-mean, unit-variance entries has variance 1, and the variances of independent terms add, giving $d_k$. So the standard deviation is $\sqrt{d_k}$, which is exactly what we divide by.
3. $10^{10}$ scores × 4 bytes = **40 GB** per head per layer. That's why FlashAttention (M5.4) computes attention in tiles without ever storing the full matrix.

**Next:** M1.2, multi-head attention. Run several of these in parallel, each with its own $W_Q, W_K, W_V$.